# Séance 4 — Trois modèles qui reconnaissent des fleurs

**QUID : peut-on reconnaître l'espèce d'un iris à partir de quatre mesures ?**

On connaît les mesures et l'espèce de 150 fleurs. On donne une partie de ces exemples aux modèles, puis on leur demande de reconnaître des fleurs qu'ils n'ont jamais vues.

**Objectif :** comprendre les étapes d'une classification supervisée et constater qu'un modèle peut faire mieux qu'une règle naïve.

Ce notebook est autonome : aucune connexion, aucun fichier à télécharger, aucun notebook précédent à exécuter. Iris est un petit jeu de données pédagogique inclus dans scikit-learn. Ses trois espèces sont relativement faciles à distinguer ; ses résultats ne représentent pas la difficulté de tous les problèmes réels.

## 1. Observer les exemples

Chaque ligne représente une fleur. Les quatre entrées sont la longueur et la largeur du sépale et du pétale, en centimètres. La réponse à apprendre est l'une des trois espèces : setosa, versicolor ou virginica.

**À dire :** « Les mesures sont les indices ; l'espèce est la réponse. »

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

iris = load_iris(as_frame=True)
X = iris.data.rename(columns={
    "sepal length (cm)": "longueur_sepale", "sepal width (cm)": "largeur_sepale",
    "petal length (cm)": "longueur_petale", "petal width (cm)": "largeur_petale",
})
y = iris.target
species = list(iris.target_names)
flowers = X.copy()
flowers["espece"] = y.map(dict(enumerate(species)))
display(flowers.head())
print(f"{len(flowers)} fleurs, {X.shape[1]} mesures, {y.nunique()} espèces")
display(flowers.espece.value_counts().rename("nombre de fleurs"))

## 2. Mettre des fleurs de côté avant d'apprendre

Nous réservons 30 % des fleurs au test. `stratify=y` conserve la proportion des trois espèces. `random_state=42` permet de retrouver le même découpage en relançant le notebook.

Le test reste à l'écart de l'entraînement. Nous fixons les modèles et leurs paramètres à l'avance : nous ne les modifions pas pour améliorer ce score.

**À dire :** « On ne donne pas les questions de l'examen pendant l'entraînement. »

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
assert set(X_train.index).isdisjoint(X_test.index)
print(f"Entraînement : {len(X_train)} fleurs ; test : {len(X_test)} fleurs")
display(pd.DataFrame({"entraînement": y_train.value_counts(),
                      "test": y_test.value_counts()}).rename(index=dict(enumerate(species))))

fig, ax = plt.subplots(figsize=(7, 4))
for label, name in enumerate(species):
    rows = X_train.loc[y_train == label]
    ax.scatter(rows.longueur_petale, rows.largeur_petale, label=name, alpha=0.75)
ax.set(xlabel="Longueur du pétale (cm)", ylabel="Largeur du pétale (cm)",
       title="Fleurs d'entraînement : les mesures donnent déjà des indices")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Faire apprendre trois modèles

- **Arbre de décision :** pose des questions successives sur les mesures.
- **5 plus proches voisins :** regarde les cinq fleurs d'entraînement les plus proches et fait voter leurs espèces.
- **Régression logistique :** combine les mesures pour attribuer une probabilité à chaque espèce. Malgré son nom, c'est ici un modèle de classification.

La référence naïve prédit toujours la même espèce. Comme le test contient autant de fleurs de chaque espèce, elle obtient environ 33 % de bonnes réponses.

La standardisation met les mesures sur une échelle comparable pour les voisins et la régression. Elle est apprise uniquement sur le train grâce aux pipelines.

In [ ]:
models = {
    "Référence : toujours la même espèce": DummyClassifier(strategy="most_frequent"),
    "Arbre de décision (profondeur 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "5 plus proches voisins": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({"modèle": name, "bonnes réponses": int((pred == y_test).sum()),
                    "fleurs testées": len(y_test), "accuracy (%)": round(100 * accuracy_score(y_test, pred), 1)})
results = pd.DataFrame(results)
display(results)
ax = results.plot.barh(x="modèle", y="accuracy (%)", legend=False, figsize=(8, 4))
ax.set(xlim=(0, 105), xlabel="Bonnes réponses (%)", ylabel="", title="Même examen pour tous les modèles")
plt.tight_layout()
plt.show()

## 4. Voir les erreurs

Une matrice de confusion compare les vraies espèces (lignes) aux espèces prédites (colonnes). Les bonnes réponses sont sur la diagonale ; les autres cases sont les erreurs.

**À demander :** « Quelles espèces sont les plus faciles à confondre ? »

Les trois modèles sont affichés, sans choisir un gagnant sur ce petit test.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, list(models)[1:]):
    ConfusionMatrixDisplay.from_predictions(y_test, predictions[name], labels=[0, 1, 2],
        display_labels=species, ax=ax, colorbar=False)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 5. Lire les règles de l'arbre

Chaque question utilise une mesure. On descend à gauche si la condition est vraie, à droite sinon, jusqu'à une prédiction.

`class` indique l'espèce majoritaire au nœud ; `samples` compte les fleurs d'entraînement qui arrivent à ce nœud ; `value` donne leurs effectifs par espèce. L'arbre utilise les quatre mesures, mais peut n'en retenir que certaines pour ses questions.

In [ ]:
tree = models["Arbre de décision (profondeur 3)"]
fig, ax = plt.subplots(figsize=(15, 7))
plot_tree(tree, feature_names=list(X.columns), class_names=species,
          filled=True, rounded=True, impurity=False, ax=ax, fontsize=10)
plt.tight_layout()
plt.show()

## 6. Prédire une fleur mise de côté

On prend la première fleur du test. Les modèles ont déjà terminé leur apprentissage : aucun `fit` supplémentaire n'est nécessaire.

**À dire :** « Maintenant on fournit seulement les mesures. Le modèle propose une espèce, puis on vérifie avec la réponse connue. »

In [ ]:
flower = X_test.iloc[[0]]
display(flower)
for name, model in list(models.items())[1:]:
    print(f"{name} : {species[int(model.predict(flower)[0])]}")
print("Espèce réelle :", species[int(y_test.iloc[0])])

## Ce qu'il faut retenir

1. On définit une question et une réponse à prédire.
2. On sépare les exemples d'entraînement et de test avant d'apprendre.
3. On compare plusieurs modèles à une référence simple sur les mêmes fleurs.
4. On regarde aussi les erreurs, pas seulement le score.

Ici, les mesures permettent de reconnaître les espèces bien mieux que la référence naïve. Cela ne prouve pas qu'un modèle gagnera toujours : le test ne contient que 45 fleurs, et changer le découpage peut changer les scores. Une seule réponse change l'accuracy d'environ 2,2 points. Ces exemples ne justifient donc pas un classement définitif des algorithmes.

**Lien avec le notebook Google Trends :** ici, chaque espèce est représentée dans le test et les modèles sont comparés à une référence sur une question simple. Dans une série temporelle, il faut conserver l'ordre du temps et vérifier ce que contiennent réellement les périodes de validation.